# Ścieżka C · Ekwiwalent pewności loterii 50/50

**Projektowanie i badanie informatycznych systemów wspomagania decyzji** · notatnik projektu z aplikacją · szkielet do uzupełnienia

Notatnik prowadzi cały projekt ścieżki C w jednym miejscu, zgodnie z przewodnikiem *C — Certainty Equivalent — Przewodnik*:

| Etap | Kto robi | Pliki |
|---|---|---|
| 1–3 · eksport, 27 wyborów, przedziały CE i q | zespół (luki `# TODO`) | `qualtrics_raw.csv` → `responses_clean.csv` → `profil.csv` |
| 4–5 · interpolacja i nowe decyzje S1/S2 | zespół (luki `# TODO`) | `cases.csv` |
| 6 · zamrożenie, kontrakt, aplikacja | gotowe | `session_config.json`, `app.py` |
| 7 · analiza walidacji i wrażliwości skali | zespół | wynik do raportu |

**Aplikacja jest gotowa.** Nie piszecie interfejsu. Aplikacja czyta pliki tworzone przez ten notatnik i działa tylko wtedy, gdy spełniają one kontrakt.

**Dwa tryby pracy** (komórka konfiguracji poniżej):
- `TRYB = 'przyklad'`: wymyślony uczestnik P01 z przewodnika. Testy sprawdzają liczby kontrolne z przewodnika. Zacznij tutaj.
- `TRYB = 'wlasne'`: eksport Waszego uczestnika. Testy funkcji działają dalej, a testy liczb P01 są pomijane: notatnik wypisuje wtedy Wasz wynik.

Notatnik używa **gotowego wariantu** doboru kwot (reguła S1/S2, 12 decyzji). Przewodnik dopuszcza też 8–12 kwot dobranych przez zespół przed sesją. Wtedy wystarczy podmienić listę bodźców w sekcji 5.

**Pomoc AI:** [Konsultant DSS (Gem w Gemini)](https://gemini.google.com/gem/1B6czvZbCeBBuqqjSMid08BPCGsWN1qDe?usp=sharing) wyjaśnia pojęcia i błędy, pomaga zaplanować testy, ale nie uzupełnia luk `# TODO`. Każde użycie AI zapiszcie w dzienniku: cel pomocy, decyzja zespołu, sposób kontroli. Plan pracy i przykłady P01: [strona Pracowni](https://p4welrog.github.io/decyzje/).

> **Jak pracować.** Uruchamiaj komórki po kolei. W miejscach `# TODO` uzupełnij kod i usuń `raise NotImplementedError`. Test pod każdym krokiem pokazuje ✓ albo zatrzymuje się na ✗ z opisem. Nie zmieniaj liczb w testach.

## 0 · Konfiguracja i folder projektu na Dysku Google
W Colabie pliki znikają po rozłączeniu, dlatego folder roboczy leży na Dysku Google. Przy pierwszym uruchomieniu Colab poprosi o zgodę na dostęp do Dysku. Folder można udostępnić zespołowi i prowadzącym. Dla przykładu i dla każdego uczestnika powstaje osobny folder.

W plikach są tylko kod uczestnika i jego odpowiedzi. Informacja przed zgodą powinna mówić, gdzie przechowujecie dane i kto ma do nich dostęp.

In [ ]:
TRYB = 'przyklad'          # 'przyklad' albo 'wlasne'
PARTICIPANT = 'P01'        # kod uczestnika (taki sam w Qualtricsie i w sesji)
SESSION_ID = 'S01'         # identyfikator sesji walidacyjnej
INSTRUCTOR_PIN = '2468'    # kod widoku prowadzącego w aplikacji; nie pokazuj go uczestnikowi
UZYJ_DYSKU = True          # w Colabie: folder projektu na Dysku Google

import os, sys, hashlib, inspect, json, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if TRYB not in ('przyklad', 'wlasne'):
    raise ValueError("TRYB musi mieć wartość 'przyklad' albo 'wlasne'")
if TRYB == 'przyklad':
    PARTICIPANT = 'P01'
if IN_COLAB and UZYJ_DYSKU:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/projekt_C'
else:
    ROOT = os.path.abspath('projekt_C')
WORKDIR = os.path.join(ROOT, 'przyklad_P01' if TRYB == 'przyklad' else PARTICIPANT)
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

def check(label, cond):
    """Test ogólny: działa w obu trybach."""
    if not bool(cond):
        raise AssertionError('✗ ' + label)
    print('✓', label)

def check_example(label, cond, value=None):
    """Test liczb przykładu P01: w trybie 'wlasne' tylko wypisuje Wasz wynik."""
    if TRYB == 'przyklad':
        check(label, cond)
    else:
        print('ℹ pominięto test przykładu:', label, '' if value is None else f'| Wasz wynik: {value}')

def close(a, b, tol=1e-6):
    return a is not None and b is not None and abs(a - b) <= tol

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def file_sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def write_qualtrics_csv(path, rows, descriptions):
    """Plik w układzie eksportu Qualtricsa: 3 wiersze nagłówkowe + odpowiedzi."""
    cols = list(rows[0].keys())
    df = pd.DataFrame([[descriptions.get(c, c) for c in cols], [json.dumps({'ImportId': f'QID_{c}'}) for c in cols]]
                      + [[r.get(c, '') for c in cols] for r in rows], columns=cols)
    df.to_csv(path, index=False)

print(f'Tryb: {TRYB} | uczestnik: {PARTICIPANT} | folder: {WORKDIR}')

## 1 · Stałe procedury (rozdz. 3)
Trzy listy dla stawek X = 100, 1000 i 10 000 zł, po 9 wierszy: loteria „50% X zł, 50% 0 zł” albo kwota pewna 10–90% X. Kody odpowiedzi: 1 = LOTTERY, 2 = SURE.

In [ ]:
PROTOCOL = 'C_v1'
MODEL_VERSION = 'C_interp_v1'
LOTTERY, SURE = 1, 2
STAKES = [100, 1000, 10000]
FRACTIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
P_WIN = 0.5
TIE_EPS = 0.01                       # |CE_est − S| < 0,01 zł → TIE
INTERP_SCALE = 'stake'               # decyzja projektowa ustalona PRZED walidacją: 'stake' albo 'log_stake'
PATTERNS_P01 = {100: 'LLLLPPPPP', 1000: 'LLLPPPPPP', 10000: 'LLPPPPPPP'}

def sure_amount(X, row):
    return round(FRACTIONS[row - 1] * X, 2)

## 2 · Eksport z Qualtricsa (rozdz. 4 i 7)
Pola `C_X100_01`–`09`, `C_X1000_01`–`09` i `C_X10000_01`–`09`. Sufiks to numer wiersza listy, a nie kwota.

In [ ]:
FIELDS_P01 = {f'C_X{X}_{r:02d}': (LOTTERY if PATTERNS_P01[X][r - 1] == 'L' else SURE) for X in STAKES for r in range(1, 10)}

def make_example_raw(path):
    base = dict(ResponseId='R_P01', participant_id='P01', project_path='C', protocol_version=PROTOCOL, phase='profile',
                Status=0, Finished=1, Progress=100, **{'Duration (in seconds)': 388}, zgoda=1)
    desc = {f'C_X{X}_{r:02d}': f'Loteria 50% {X} zł / 50% 0 zł czy {sure_amount(X, r):g} zł na pewno?' for X in STAKES for r in range(1, 10)}
    write_qualtrics_csv(path, [{**base, 'ResponseId': 'R_prev01', 'Status': 1, **FIELDS_P01}, {**base, **FIELDS_P01}], desc)

In [ ]:
RAW_PATH = 'qualtrics_raw.csv'
if TRYB == 'przyklad':
    make_example_raw(RAW_PATH)
elif not os.path.exists(RAW_PATH):
    if not IN_COLAB:
        raise FileNotFoundError(f'Umieść eksport jako {os.path.join(WORKDIR, RAW_PATH)}')
    from google.colab import files
    up = files.upload()
    open(RAW_PATH, 'wb').write(next(iter(up.values())))
print('Plik źródłowy:', os.path.join(WORKDIR, RAW_PATH))

In [ ]:
def load_raw(path):
    """Eksport Qualtricsa: trzy wiersze nagłówkowe, dane od czwartego."""
    return pd.read_csv(path, skiprows=[1, 2])

def select_response(raw, participant_id, protocol_version, phase='profile'):
    """Dokładnie jedno wypełnienie: Status 0 (nie Preview 1, nie Test 2), Finished 1, zgoda 1, kod, wersja, faza."""
    # TODO: przefiltruj raw według sześciu warunków z opisu, sprawdź, że został dokładnie jeden wiersz, zwróć go
    raise NotImplementedError

raw = load_raw(RAW_PATH)
resp = select_response(raw, PARTICIPANT, PROTOCOL)
print('Wybrano wypełnienie', resp['ResponseId'])

## 3 · Długi plik 27 wierszy i scoring list (rozdz. 5–7)
| Wzorzec | Status | Wynik |
|---|---|---|
| jedno przełączenie L → P | `valid` | `ce_low` = ostatnia L, `ce_high` = pierwsza P, `ce_mid` = środek |
| powrót do L albo start od P | `nonmonotonic` | brak jednego CE |
| same L | `lower_bound` | CE > 0,9X (bez wymyślonego środka) |
| same P | `upper_bound` | CE < 0,1X |
| brak odpowiedzi | `missing` | nieznane miejsce przełączenia |

Dla listy `valid`: EV = X/2, q = ce_mid/EV, granice q_low = ce_low/EV i q_high = ce_high/EV, premia = EV − ce_mid.

In [ ]:
def to_long(row):
    """27 wierszy: participant_id, stake, row, sure_amount, choice_code (brak = None, nie 0 i nie SURE)."""
    # TODO: pętla po stawkach i wierszach 1–9; brak odpowiedzi zapisz jako None
    raise NotImplementedError

def score_list(X, codes):
    """codes: lista 9 kodów (1/2/None). Zwraca dict z polami profilu."""
    ev = X * P_WIN
    res = dict(stake=X, probability=P_WIN, ev=ev, status=None, ce_low=None, ce_high=None, ce_mid=None,
               q=None, q_low=None, q_high=None, risk_premium=None)
    # TODO: rozpoznaj pięć przypadków z tabeli; dla valid wypełnij granice, środek, q, q_low, q_high i premię
    raise NotImplementedError
    return res

def build_profile(long):
    """Gotowe: jeden wiersz na stawkę."""
    return pd.DataFrame([score_list(X, list(long[long.stake == X].sort_values('row').choice_code)) for X in STAKES])

codes = lambda p: [LOTTERY if ch == 'L' else SURE if ch == 'P' else None for ch in p]
PROFILE_P01 = build_profile(to_long(FIELDS_P01))
check('dane P01: X = 100 → CE 40–50, środek 45, q = 0,9', PROFILE_P01.iloc[0][['ce_low', 'ce_high', 'ce_mid']].tolist() == [40, 50, 45])
check('dane P01: q = 0,9 / 0,7 / 0,5 i premie 5 / 150 / 2500', all(close(a, b) for a, b in zip(PROFILE_P01.q, [0.9, 0.7, 0.5]))
      and all(close(a, b) for a, b in zip(PROFILE_P01.risk_premium, [5, 150, 2500])))
check('L P L P P … → nonmonotonic', score_list(100, codes('LPLPPPPPP'))['status'] == 'nonmonotonic')
check('P P L L L … → nonmonotonic', score_list(100, codes('PPLLLLLLL'))['status'] == 'nonmonotonic')
check('same L → lower_bound, same P → upper_bound', (score_list(100, codes('LLLLLLLLL'))['status'], score_list(100, codes('PPPPPPPPP'))['status']) == ('lower_bound', 'upper_bound'))
check('brak odpowiedzi → missing', score_list(100, codes('LLL-PPPPP'))['status'] == 'missing')

long = to_long(resp)
long.to_csv('responses_clean.csv', index=False)
profile = build_profile(long)
profile['interp_scale'] = INTERP_SCALE
check_example('profil uczestnika = P01 (q 0,9 / 0,7 / 0,5)', list(profile.q) == list(PROFILE_P01.q), list(profile.q))
profile.insert(0, 'participant_id', PARTICIPANT)
profile.to_csv('profil.csv', index=False)
profile

## 4 · Reguła systemu i interpolacja (rozdz. 8)
$q(X) = q_a + \frac{X - X_a}{X_b - X_a}(q_b - q_a)$ między dwiema najbliższymi stawkami profilu (skala `stake`). W wariancie `log_stake` interpolujemy po log10 X. Ta sama interpolacja daje granice z `q_low` i `q_high`. CE_est = EV · q(X).

Funkcja **odmawia predykcji** (`NO_PREDICTION`), gdy p ≠ 0,5, gdy X leży poza zakresem 100–10 000 zł albo gdy któryś punkt profilu ma status inny niż `valid`. Kwota pewna wewnątrz granic CE dostaje flagę niepewności siatki.

In [ ]:
def interp_q(X, profile, key='q', scale=INTERP_SCALE):
    """Interpolowana wartość profile[key] dla stawki X albo None poza zakresem."""
    # TODO: znajdź przedział [X_a, X_b] zawierający X i zastosuj wzór (dla log_stake na log10)
    raise NotImplementedError

def predict_lottery(X, S, profile, p=P_WIN, scale=INTERP_SCALE, eps=TIE_EPS):
    """dict(pred, ce, ce_low, ce_high, uncertain, reason) – pred ∈ LOTTERY, SURE, TIE, NO_PREDICTION."""
    out = dict(pred='NO_PREDICTION', ce=None, ce_low=None, ce_high=None, uncertain=False, reason='')
    # TODO: trzy warunki odmowy, potem q, EV, CE i granice; porównanie CE z S
    raise NotImplementedError
    return out

def predict_ev(X, S, p=P_WIN, eps=TIE_EPS):
    """Baseline: wybór większej EV (porównanie, nie etykieta poprawności). Gotowe."""
    ev = X * p
    return 'TIE' if abs(ev - S) < eps else ('LOTTERY' if ev > S else 'SURE')

def explain_lottery(X, S, profile, scale=INTERP_SCALE):
    """Wyjaśnienie pokazywane po serii. Gotowe."""
    r = predict_lottery(X, S, profile, scale=scale)
    if r['ce'] is None:
        return f'Model nie stosuje profilu: {r["reason"]}.'
    rel = 'wyższa' if S > r['ce'] else 'niższa' if S < r['ce'] else 'równa'
    t = (f'Według Twojego profilu loteria 50% na {X} zł i 50% na 0 zł jest dla Ciebie warta około {r["ce"]:.2f} zł na pewno '
         f'(zakres {r["ce_low"]:.2f}–{r["ce_high"]:.2f} zł wynika z siatki odpowiedzi). Kwota pewna {S} zł jest od tego {rel}.')
    return t + (' Ta kwota leży w obszarze niepewności siatki.' if r['uncertain'] else '')

r = predict_lottery(2000, 600, PROFILE_P01)
check('P01, X = 2000: CE = 677,78 zł, granice 577,78–777,78', round(r['ce'], 2) == 677.78 and (round(r['ce_low'], 2), round(r['ce_high'], 2)) == (577.78, 777.78))
check('600 zł → LOTTERY w obszarze niepewności; 800 zł → SURE', r['pred'] == 'LOTTERY' and r['uncertain'] and predict_lottery(2000, 800, PROFILE_P01)['pred'] == 'SURE')
check('skala log_stake: CE = 639,79 zł', round(predict_lottery(2000, 600, PROFILE_P01, scale='log_stake')['ce'], 2) == 639.79)
check('X = 50 000 albo p = 0,2 → odmowa', predict_lottery(50000, 1000, PROFILE_P01)['pred'] == predict_lottery(2000, 600, PROFILE_P01, p=0.2)['pred'] == 'NO_PREDICTION')
check('S = punktowy CE → TIE', predict_lottery(2000, 677.7777777777778, PROFILE_P01)['pred'] == 'TIE')
_bad = PROFILE_P01.copy(); _bad.loc[1, 'status'] = 'nonmonotonic'
check('punkt nonmonotonic → brak personalizacji', predict_lottery(2000, 600, _bad)['pred'] == 'NO_PREDICTION')

## 5 · Gotowy wariant: 12 nowych decyzji ze stałej reguły S1/S2 (rozdz. 11)
Stawki 200, 500, 2000, 3000, 5000 i 8000 zł, przy każdej dwie kwoty pewne:
- **S1** = 80% dolnej granicy przeniesionego CE. Sprawdza przewidywanie wspólne z regułą EV.
- **S2** = środek między górną granicą CE a EV. Rozróżnia oba modele.

Obie kwoty zaokrąglamy w dół: do 1 zł przy stawkach do 500 zł i do 100 zł przy wyższych. LEFT oznacza kwotę pewną, RIGHT loterię.

In [ ]:
VAL_STAKES = [200, 500, 2000, 3000, 5000, 8000]

def sure_amounts(X, profile, scale=INTERP_SCALE):
    """Zwraca (S1, S2, uwaga)."""
    # TODO: granice CE z predict_lottery, krok zaokrąglenia, S1 i S2 z math.floor
    raise NotImplementedError

def build_cases(profile):
    rows = []
    for X in VAL_STAKES:
        s1, s2, note = sure_amounts(X, profile)
        for lab, S in (('S1', s1), ('S2', s2)):
            p = predict_lottery(X, S, profile)
            rows.append(dict(case_id=f'C{len(rows) + 1:02d}', stake=X, probability=P_WIN, sure_amount=S, amount_type=lab,
                             ce_est=p['ce'], ce_low=p['ce_low'], ce_high=p['ce_high'], uncertain=p['uncertain'],
                             pred_personal=p['pred'], pred_baseline=predict_ev(X, S),
                             pred_log_stake=predict_lottery(X, S, profile, scale='log_stake')['pred'],
                             explanation=explain_lottery(X, S, profile), note=note if lab == 'S2' else ''))
    return pd.DataFrame(rows)

_c = build_cases(PROFILE_P01)
check('P01: kwoty 62/98, 142/238, 400/800, 600/1300, 1000/2100, 1400/3200', list(_c.sure_amount) == [62, 98, 142, 238, 400, 800, 600, 1300, 1000, 2100, 1400, 3200])
check('P01: S1 → oba LOTTERY; S2 → profil SURE, EV LOTTERY', all(_c[_c.amount_type == 'S1'].pred_personal == 'LOTTERY')
      and all(_c[_c.amount_type == 'S2'].pred_personal == 'SURE') and all(_c.pred_baseline == 'LOTTERY'))

PROFILE_OK = bool((profile.status == 'valid').all())
PROFILE_NOTE = 'valid' if PROFILE_OK else 'statusy list: ' + ', '.join(profile.status)
cases = build_cases(profile) if PROFILE_OK else pd.DataFrame(columns=_c.columns)
rng = np.random.default_rng(20261020)
cases['display_order'] = rng.permutation(len(cases)) + 1
cases['left_text'] = cases.sure_amount.map(lambda s: f'### {s:g} zł na pewno')
cases['right_text'] = cases.stake.map(lambda X: f'### Loteria: 50% szans na {X} zł i 50% szans na 0 zł')
cases['left_code'], cases['right_code'] = 'SURE', 'LOTTERY'
cases['model_version'] = MODEL_VERSION
cases['params'] = ('q=' + '/'.join(f'{v:.4f}' for v in profile.q) if PROFILE_OK else 'q=None') + f'|scale={INTERP_SCALE}|eps={TIE_EPS}'
SESSION_EXTRA = dict(title='Pewna kwota czy loteria?', left_button='Wybieram kwotę pewną', right_button='Wybieram loterię',
                     code_labels={'SURE': 'kwota pewna', 'LOTTERY': 'loteria'},
                     intro='W każdej sytuacji wybierz: określona kwota na pewno albo loteria, w której z szansą 50% dostajesz podaną kwotę, '
                           'a z szansą 50% dostajesz 0 zł. Decyzje są hipotetyczne. Nie ma poprawnych odpowiedzi.')
cases.sort_values('display_order')[['display_order', 'case_id', 'stake', 'sure_amount', 'amount_type', 'pred_personal', 'pred_baseline']]

## 6 · Zamrożenie, kontrakt i aplikacja

In [ ]:
# Opcjonalnie (rozszerzenie, np. własny interfejs): zapis przetestowanych funkcji do model.py.
# Aplikacja kursu nie potrzebuje tego pliku: opisy, predykcje i wyjaśnienia są w cases.csv.
EKSPORT_MODELU = False
if EKSPORT_MODELU:
    FUNCS = [sure_amount, to_long, score_list, build_profile, interp_q, predict_lottery, predict_ev, explain_lottery, sure_amounts]
    CONSTS = ['LOTTERY', 'SURE', 'STAKES', 'FRACTIONS', 'P_WIN', 'TIE_EPS', 'INTERP_SCALE', 'MODEL_VERSION']
    src = ['"""model.py – wygenerowane z notatnika ścieżki C; nie edytuj ręcznie."""', 'import math', 'import pandas as pd', '']
    src += [f'{n} = {globals()[n]!r}' for n in CONSTS] + [''] + [inspect.getsource(f) for f in FUNCS]
    open('model.py', 'w', encoding='utf-8').write('\n'.join(src))
    sys.path.insert(0, WORKDIR)
    if 'model' in sys.modules: del sys.modules['model']
    import model
    check('model.py: CE(2000) = 677,78', round(model.predict_lottery(2000, 600, PROFILE_P01)['ce'], 2) == 677.78)

## Zamrożenie przed sesją
Komórka zapisuje `cases.csv` z opisami opcji i predykcjami obu modeli oraz `session_config.json` ze skrótem SHA-256. Skrót wpisz do dziennika **przed sesją**. W trybie `wlasne` notatnik nie nadpisze zamrożonych przypadków: jeśli po zamrożeniu zmienisz dane albo kod, komórka zatrzyma się z błędem. 

In [ ]:
new_csv = cases.to_csv(index=False).encode()
new_hash = hashlib.sha256(new_csv).hexdigest()
if not PROFILE_OK:
    raise RuntimeError(f'Profil nie pozwala na personalizację ({PROFILE_NOTE}). Nie zamrażamy przypadków i nie prowadzimy sesji.')
if TRYB == 'wlasne' and os.path.exists('session_config.json'):
    frozen = json.load(open('session_config.json', encoding='utf-8'))
    if frozen['frozen_sha256'] != new_hash:
        raise RuntimeError('Przypadki są już zamrożone, a nowe obliczenia dają inny cases.csv. Nie nadpisuję. '
                           'Jeśli zmiana nastąpiła PRZED sesją, usuń świadomie session_config.json i opisz to w dzienniku; '
                           'po sesji wymaga to nowej wersji modelu i nowego zestawu.')
    print('Zamrożenie bez zmian:', new_hash)
else:
    open('cases.csv', 'wb').write(new_csv)
    json.dump(dict(contract='C_v1', participant_id=PARTICIPANT, session_id=SESSION_ID, frozen_sha256=new_hash,
                   frozen_at=now_iso(), model_version=MODEL_VERSION, seed=20261020, instructor_pin=INSTRUCTOR_PIN,
                   profile_ok=PROFILE_OK, profile_note=PROFILE_NOTE, **SESSION_EXTRA),
              open('session_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('Zamrożono cases.csv | SHA-256:', new_hash)
FROZEN_HASH = file_sha256('cases.csv')

In [ ]:
cfg = json.load(open('session_config.json', encoding='utf-8'))
_cases = pd.read_csv('cases.csv')
missing = [c for c in ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code', 'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation'] if c not in _cases.columns]
check('cases.csv: wymagane kolumny', not missing or print('Brakuje:', missing))
check('session_config.json: kontrakt i zgodny skrót cases.csv', cfg['contract'] == 'C_v1' and cfg['frozen_sha256'] == file_sha256('cases.csv'))
check('profil.csv istnieje', os.path.exists('profil.csv'))

## Aplikacja i sesja z uczestnikiem
Komórka z kodem aplikacji jest zwinięta. **Nie edytuj jej.** Zapisuje plik `app.py` w folderze projektu.

**Przebieg sesji:**
1. Uczestnik widzi „kwota pewna / loteria 50% X zł i 50% 0 zł” i wybiera bez podpowiedzi. Każdy wybór trafia od razu do `validation.csv`.
2. Po wszystkich wyborach aplikacja pokazuje rekomendacje i wyjaśnienia z zakresem niepewności CE, a potem zbiera jedną ocenę dopasowania i jedną ocenę użyteczności serii (1–7).

Widok prowadzącego (panel boczny, kod `INSTRUCTOR_PIN`) pokazuje kontrakt, zamrożone przypadki i log. Nie otwieraj go przy uczestniku. Odświeżenie strony nie gubi danych: aplikacja wznawia sesję od pierwszego niezapisanego kroku.

In [ ]:
#@title Kod aplikacji – gotowy, nie edytuj { display-mode: "form" }
APP_SOURCE = r'''"""Sesja walidacyjna – ścieżki A, B, C i E (wybór jednej z dwóch opcji).

Gotowa aplikacja kursu: zespół jej NIE edytuje. Aplikacja niczego nie przewiduje sama:
opisy opcji, predykcje i wyjaśnienia pochodzą z zamrożonego cases.csv utworzonego w notatniku.

Uruchomienie:  streamlit run app.py -- --data <folder z plikami projektu>
Folder musi zawierać: cases.csv, profil.csv i session_config.json (tworzy je notatnik).
Aplikacja zapisuje: validation.csv (po każdym wyborze) i session_<id>.json (oceny serii).
"""
import argparse
import hashlib
import json
import os
import sys
from datetime import datetime, timezone

import pandas as pd
import streamlit as st

APP_VERSION = 'pred_app_v1'
REQUIRED_CASES = ['case_id', 'display_order', 'left_text', 'right_text', 'left_code', 'right_code',
                  'pred_personal', 'pred_baseline', 'model_version', 'params', 'explanation']
REQUIRED_CFG = ['contract', 'participant_id', 'session_id', 'frozen_sha256', 'instructor_pin', 'profile_ok',
                'title', 'intro', 'left_button', 'right_button', 'code_labels']


def data_dir():
    p = argparse.ArgumentParser()
    p.add_argument('--data', default='.')
    return os.path.abspath(p.parse_known_args(sys.argv[1:])[0].data)


DATA = data_dir()
path = lambda name: os.path.join(DATA, name)
now_iso = lambda: datetime.now(timezone.utc).isoformat(timespec='seconds')
sha256 = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()


def load_project():
    """Wczytuje pliki i sprawdza kontrakt. Zwraca (cfg, cases, błędy, ostrzeżenia)."""
    errors, warnings = [], []
    for name in ['cases.csv', 'profil.csv', 'session_config.json']:
        if not os.path.exists(path(name)):
            errors.append(f'Brak pliku {name} w folderze {DATA}. Uruchom notatnik do sekcji „Zamrożenie”.')
    if errors:
        return {}, None, errors, warnings
    cfg = json.load(open(path('session_config.json'), encoding='utf-8'))
    missing = [k for k in REQUIRED_CFG if k not in cfg]
    if missing:
        errors.append(f'session_config.json: brak pól {", ".join(missing)}')
    cases = pd.read_csv(path('cases.csv'), dtype={'case_id': str})
    missing = [c for c in REQUIRED_CASES if c not in cases.columns]
    if missing:
        errors.append(f'cases.csv: brak kolumn {", ".join(missing)}')
    if sha256(path('cases.csv')) != cfg.get('frozen_sha256'):
        errors.append('cases.csv różni się od wersji zamrożonej (SHA-256 nie zgadza się z session_config.json).')
    if not cfg.get('profile_ok', False):
        errors.append(f'Profil nie pozwala na personalizację: {cfg.get("profile_note", "")}')
    if not errors:
        bad = set(cases.pred_personal.astype(str)) - set(cfg['code_labels']) - {'TIE', 'NO_PREDICTION'}
        if bad:
            errors.append(f'cases.csv: nieznane kody predykcji {bad}')
        cases = cases.sort_values('display_order').reset_index(drop=True)
    return cfg, cases, errors, warnings


def read_log():
    if os.path.exists(path('validation.csv')) and os.path.getsize(path('validation.csv')) > 0:
        return pd.read_csv(path('validation.csv'), dtype={'session_id': str, 'case_id': str})
    return pd.DataFrame(columns=['session_id', 'case_id'])


def append_log(row):
    log = pd.concat([read_log(), pd.DataFrame([row])], ignore_index=True)
    tmp = path('validation.csv.tmp')
    log.to_csv(tmp, index=False)
    os.replace(tmp, path('validation.csv'))       # zapis atomowy


st.set_page_config(page_title='Sesja walidacyjna', page_icon='🧭', layout='wide')
cfg, cases, errors, warnings = load_project()
SESSION = str(cfg.get('session_id', 'S01'))
SESSION_JSON = path(f'session_{SESSION}.json')

with st.sidebar:
    st.caption(f'{APP_VERSION} · kontrakt {cfg.get("contract", "?")}')
    pin = st.text_input('Kod prowadzącego', type='password')
    instructor = bool(cfg) and pin != '' and pin == str(cfg.get('instructor_pin'))

if instructor:
    st.title('Widok prowadzącego')
    for e in errors:
        st.error(e)
    for w in warnings:
        st.warning(w)
    if not errors:
        st.success(f'Kontrakt spełniony · cases.csv zgodny z zamrożeniem · SHA-256 {cfg["frozen_sha256"][:16]}…')
        st.subheader('Profil')
        st.dataframe(pd.read_csv(path('profil.csv')), hide_index=True)
        st.subheader('Zamrożone przypadki')
        st.dataframe(cases[['display_order', 'case_id', 'left_text', 'right_text', 'pred_personal', 'pred_baseline']], hide_index=True)
    log = read_log()
    n = int((log.session_id.astype(str) == SESSION).sum()) if len(log) else 0
    st.subheader(f'Log sesji {SESSION}: {n} zapisanych wyborów')
    st.dataframe(log, hide_index=True)
    if os.path.exists(path('validation.csv')):
        st.download_button('Pobierz validation.csv', open(path('validation.csv'), 'rb').read(), 'validation.csv', 'text/csv')
    if os.path.exists(SESSION_JSON):
        st.json(json.load(open(SESSION_JSON, encoding='utf-8')))
    st.stop()

st.title(cfg.get('title', 'Sesja walidacyjna'))
if errors:
    st.error('Aplikacja nie jest gotowa do sesji. Poproś prowadzącego o sprawdzenie (widok prowadzącego w panelu bocznym).')
    st.stop()

log = read_log()
done = log[log.session_id.astype(str) == SESSION]
remaining = cases[~cases.case_id.isin(set(done.case_id.astype(str)))]
if os.path.exists(SESSION_JSON):
    st.success('Sesja zakończona. Dziękujemy za udział!')
    st.caption('Dla prowadzącego: próbną sesję można powtórzyć komórką „↺ Próbna sesja od początku” w notatniku.')
    st.stop()

if 'started' not in st.session_state:
    st.session_state.started = len(done) > 0
if not st.session_state.started:
    st.write(cfg['intro'])
    st.write(f'Zobaczysz {len(cases)} sytuacji. W każdej wybierz jedną z dwóch możliwości. '
             'Wyjaśnienia działania systemu zobaczysz dopiero po zapisaniu wszystkich wyborów.')
    if st.button('Rozpocznij', type='primary'):
        st.session_state.started = True
        st.session_state.started_at = now_iso()
        st.rerun()
    st.stop()

if len(remaining):
    row = remaining.iloc[0]
    k = len(done) + 1
    st.progress((k - 1) / len(cases), text=f'Sytuacja {k} z {len(cases)}')
    st.caption(cfg['intro'])
    cl, cr = st.columns(2)
    for col, txt in ((cl, row.left_text), (cr, row.right_text)):
        with col:
            with st.container(border=True):
                st.markdown(txt)

    def choose(position):
        code = row.left_code if position == 'left' else row.right_code
        rec = dict(participant_id=cfg['participant_id'], session_id=SESSION, case_id=str(row.case_id),
                   display_order=int(row.display_order), chosen_position=position, answer=code, answered_at=now_iso(),
                   model_version=row.model_version, params=row.params, app_version=APP_VERSION)
        rec.update({c: row[c] for c in cases.columns if c.startswith('pred_')})
        append_log(rec)

    b1, b2 = st.columns(2)
    b1.button(cfg['left_button'], key=f'L_{row.case_id}', on_click=choose, args=('left',), use_container_width=True)
    b2.button(cfg['right_button'], key=f'R_{row.case_id}', on_click=choose, args=('right',), use_container_width=True)
    st.stop()

st.progress(1.0, text='Wszystkie wybory zapisane')
st.header('Jak system opisuje Twoje wybory')
st.write('Teraz, po zapisaniu wszystkich decyzji, pokazujemy rekomendacje systemu i ich uzasadnienie. '
         'Rekomendacja opisuje działanie modelu, a nie poprawność Twojej decyzji.')
ans = dict(zip(done.case_id.astype(str), done.answer))
labels = cfg['code_labels']
for _, row in cases.iterrows():
    mine = ans.get(str(row.case_id))
    with st.expander(f'Sytuacja {int(row.display_order)}: wybrano „{labels.get(mine, mine)}”'):
        c1, c2 = st.columns(2)
        c1.markdown(row.left_text); c2.markdown(row.right_text)
        rec = row.pred_personal
        if rec in labels:
            st.write(f'**System rekomendowałby: {labels[rec]}.** {row.explanation}')
        else:
            st.write(f'System nie rozstrzyga tej sytuacji ({rec}). {row.explanation}')

st.header('Ocena całej serii')
fit = st.slider('Na ile rekomendacje były dopasowane do Twoich wyborów? (1 = wcale, 7 = bardzo)', 1, 7, 4)
useful = st.slider('Na ile wyjaśnienia były dla Ciebie użyteczne? (1 = wcale, 7 = bardzo)', 1, 7, 4)
if st.button('Zakończ sesję', type='primary'):
    json.dump(dict(participant_id=cfg['participant_id'], session_id=SESSION, rating_fit=int(fit), rating_usefulness=int(useful),
                   started_at=st.session_state.get('started_at', ''), finished_at=now_iso(), n_choices=len(done),
                   cases_sha256=cfg['frozen_sha256'], app_version=APP_VERSION),
              open(SESSION_JSON, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    st.rerun()
'''
open('app.py', 'w', encoding='utf-8').write(APP_SOURCE)
print('Zapisano app.py w', WORKDIR)

In [ ]:
#@title ▶ Uruchom aplikację w Colabie { display-mode: "form" }
URUCHOM = False  #@param {type:"boolean"}
import subprocess, socket, time
def _port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0
if URUCHOM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)
    if not _port_open(8501):
        subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', os.path.join(WORKDIR, 'app.py'),
                          '--server.port', '8501', '--server.headless', 'true',
                          '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
                          '--', '--data', WORKDIR],
                         stdout=open(os.path.join(WORKDIR, 'streamlit.log'), 'w'), stderr=subprocess.STDOUT)
        for _ in range(60):
            if _port_open(8501): break
            time.sleep(1)
    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(8501, anchor_text='▶ Otwórz aplikację w nowej karcie')
    else:
        print('Aplikacja: http://localhost:8501')
else:
    print('Zaznacz URUCHOM i uruchom komórkę ponownie. Jeśli link nie działa, przeprowadź sesję lokalnie (komórka niżej).')

In [ ]:
#@title ↺ Próbna sesja od początku { display-mode: "form" }
PROBA_OD_POCZATKU = False      #@param {type:"boolean"}
TO_BYLA_TYLKO_PROBA = False    #@param {type:"boolean"}
import shutil
_sid = json.load(open('session_config.json', encoding='utf-8'))['session_id']
_files = [f for f in ['validation.csv', f'session_{_sid}.json'] if os.path.exists(f)]
if not PROBA_OD_POCZATKU:
    print('Po zakończonej sesji aplikacja pokazuje tylko podziękowanie. Aby powtórzyć próbną sesję, zaznacz PROBA_OD_POCZATKU i uruchom komórkę.')
elif not _files:
    print('Brak logu sesji: aplikacja zacznie od początku.')
elif TRYB == 'wlasne' and not TO_BYLA_TYLKO_PROBA:
    print("⚠ Tryb 'wlasne': log może zawierać prawdziwą sesję uczestnika. Jeśli to była tylko próba przed sesją, "
          'zaznacz też TO_BYLA_TYLKO_PROBA. Po sesji z uczestnikiem logu nie przenosimy.')
else:
    _dst = os.path.join('proby', datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(_dst, exist_ok=True)
    for f in _files:
        shutil.move(f, os.path.join(_dst, f))
    print('Przeniesiono', ', '.join(_files), 'do', os.path.join(WORKDIR, _dst), '(nic nie usunięto). Odśwież kartę aplikacji.')
    if TRYB == 'wlasne':
        print('Zapisz w dzienniku: data, kto prowadził próbę i że log próby przeniesiono do', _dst)

**Sesja lokalna (zalecana przy uczestniku).** Colab może się rozłączyć w trakcie sesji. Pewniej jest uruchomić aplikację na laptopie:
1. Uruchom komórkę poniżej i pobierz `pakiet_sesji_C.zip` (`app.py`, `profil.csv`, `cases.csv`, `session_config.json`).
2. Na laptopie: `pip install streamlit pandas`, potem w rozpakowanym folderze `streamlit run app.py -- --data .`
3. Po sesji skopiuj `validation.csv` i `session_S01.json` do folderu projektu na Dysku i uruchom analizę w trybie `wlasne`.

In [ ]:
import zipfile
with zipfile.ZipFile('pakiet_sesji_C.zip', 'w') as z:
    for f in ['app.py', 'profil.csv', 'cases.csv', 'session_config.json']:
        z.write(f)
print('Zapisano', os.path.join(WORKDIR, 'pakiet_sesji_C.zip'))
if IN_COLAB:
    from google.colab import files
    files.download('pakiet_sesji_C.zip')

## 7 · Analiza walidacji (rozdz. 11)
- `przyklad`: log w formacie aplikacji z **syntetycznymi** odpowiedziami.
- `wlasne`: `validation.csv` i `session_S01.json` zapisane przez aplikację.

In [ ]:
UNAMBIGUOUS = lambda s: s.notna() & ~s.isin(['TIE', 'NO_PREDICTION'])

def summarize(df, pred='pred_personal', base='pred_baseline', ans='answer'):
    """Podsumowanie walidacji wg przewodnika.

    - pokrycie: liczba prób z jednoznaczną predykcją (bez TIE i NO_PREDICTION),
    - trafność obu modeli na WSPÓLNYM zbiorze prób z odpowiedzią i jednoznaczną predykcją obu,
    - podział 2×2: oba trafiły / tylko indywidualny / tylko bazowy / żaden,
    - liczba prób wspólnego zbioru, w których modele się różnią.
    """
    # TODO: 1) wybierz wspólny zbiór: odpowiedź jest, obie predykcje jednoznaczne (UNAMBIGUOUS),
    #       2) policz trafienia obu modeli na tym zbiorze i podział 2×2,
    #       3) zwróć słownik o kluczach: n_trials, n_answered, coverage_personal,
    #          coverage_baseline, n_common, hits_personal, hits_baseline, both,
    #          only_personal, only_baseline, none, n_differ.
    raise NotImplementedError

def report(s, name_p='model indywidualny', name_b='model bazowy'):
    pct = lambda k, m: f'{k}/{m} = {100*k/m:.0f}%' if m else f'{k}/{m}'
    print(f"Pokrycie: {name_p} {s['coverage_personal']}/{s['n_trials']}, {name_b} {s['coverage_baseline']}/{s['n_trials']}")
    print(f"Trafność na wspólnym zbiorze: {name_p} {pct(s['hits_personal'], s['n_common'])}, "
          f"{name_b} {pct(s['hits_baseline'], s['n_common'])}")
    print(f"2×2: oba {s['both']}, tylko indywidualny {s['only_personal']}, tylko bazowy {s['only_baseline']}, żaden {s['none']}")
    print(f"Próby wspólnego zbioru, w których modele się różnią: {s['n_differ']}/{s['n_common']}")
    print('TIE i NO_PREDICTION nie są ani trafieniem, ani pomyłką. Wynik opisuje test jednej osoby.')

In [ ]:
_t = pd.DataFrame({'pred_personal': ['A', 'B', 'TIE', 'A', 'A'],
                   'pred_baseline': ['A', 'A', 'B', 'NO_PREDICTION', 'B'],
                   'answer':        ['A', 'B', 'B', 'A', None]})
_s = summarize(_t)
check('pokrycie liczy tylko jednoznaczne predykcje', (_s['coverage_personal'], _s['coverage_baseline']) == (4, 4))
check('wspólny zbiór pomija TIE, NO_PREDICTION i brak odpowiedzi', _s['n_common'] == 2)
check('podział 2×2', (_s['both'], _s['only_personal'], _s['only_baseline'], _s['none']) == (1, 1, 0, 0))

In [ ]:
def synthetic_session():
    ans = dict(zip([f'C{i:02d}' for i in range(1, 13)], ['LOTTERY', 'SURE', 'LOTTERY', 'SURE', 'LOTTERY', 'LOTTERY',
                                                          'LOTTERY', 'SURE', 'LOTTERY', 'SURE', 'LOTTERY', 'LOTTERY']))
    v = pd.read_csv('cases.csv', dtype={'case_id': str}).sort_values('display_order')
    log = v[['case_id', 'display_order', 'model_version', 'params'] + [c for c in v.columns if c.startswith('pred_')]].copy()
    log.insert(0, 'participant_id', PARTICIPANT); log.insert(1, 'session_id', SESSION_ID)
    log['answer'] = log.case_id.map(ans)
    log['chosen_position'] = np.where(log.answer == 'SURE', 'left', 'right')
    log['answered_at'] = now_iso(); log['app_version'] = 'synthetic'
    return log, dict(rating_fit=5, rating_usefulness=6, n_choices=len(log))

In [ ]:
check('cases.csv nie zmienił się od zamrożenia', file_sha256('cases.csv') == FROZEN_HASH)
validation, session = None, None
if TRYB == 'przyklad':
    validation, session = synthetic_session()          # odpowiedzi syntetyczne w formacie logu aplikacji
elif os.path.exists('validation.csv'):
    validation = pd.read_csv('validation.csv', dtype={'session_id': str, 'case_id': str})
    validation = validation[validation.session_id == SESSION_ID]
    if os.path.exists(f'session_{SESSION_ID}.json'):
        session = json.load(open(f'session_{SESSION_ID}.json', encoding='utf-8'))
    _frozen = pd.read_csv('cases.csv', dtype={'case_id': str})
    check('log zawiera wyłącznie zamrożone przypadki', set(validation.case_id) <= set(_frozen.case_id.astype(str)))
    if 'pred_personal' in validation:
        _m = validation.merge(_frozen, on='case_id', suffixes=('', '_f'))
        check('predykcje w logu zgodne z zamrożonymi', ((_m.pred_personal == _m.pred_personal_f) & (_m.pred_baseline == _m.pred_baseline_f)).all())
    print(f'Wczytano {len(validation)} zapisów z sesji {SESSION_ID}.', '' if session else '⚠ Brak pliku session_*.json.')
else:
    print('Brak validation.csv: najpierw przeprowadź sesję.')

In [ ]:
if validation is not None:
    s = summarize(validation)
    report(s, 'model CE z profilu', 'reguła maksymalizacji EV')
    check_example('trafność 10/12 wobec 8/12, 2×2 = 6/4/2/0', (s['hits_personal'], s['hits_baseline'], s['n_common'], s['both'],
                  s['only_personal'], s['only_baseline'], s['none']) == (10, 8, 12, 6, 4, 2, 0))
    # TODO: trafienia obu modeli osobno dla S1 i S2; liczba zmian predykcji przy skali log_stake

In [ ]:
if session:
    print('Ocena dopasowania serii (1–7):', session.get('rating_fit'), '| ocena użyteczności serii (1–7):', session.get('rating_usefulness'))

## 8 · Pytania do raportu
1. Czy osoba z CE = 45 zł wybrała 45 zł? Jak raport powinien pokazać przedział i punkt roboczy?
2. Dlaczego predykcja przy 600 zł dla X = 2000 jest oznaczona jako niepewna, choć model daje jednoznaczne LOTTERY?
3. Co oznacza 10/12 wobec 8/12, skoro wszystkie predykcje EV to LOTTERY? Które próby niosą informację o personalizacji?